# Unstructured 3D Mesh: Diffusion in a Tetrahedral Mesh

This notebook demonstrates using PyFVTool's unstructured mesh support for 3D tetrahedral meshes.

## Overview

PyFVTool supports 3D unstructured meshes through `UnstructuredMesh3D`. This example shows:

1. Creating a simple tetrahedral mesh of a cube
2. Tagging boundaries by geometric location
3. Solving a steady-state diffusion equation
4. Visualizing the solution on boundary faces

## Problem Setup

We solve Laplace's equation $\nabla^2 \phi = 0$ on a unit cube $[0,1]^3$ with boundary conditions:
- Left ($x=0$): $\phi = 0$ (Dirichlet)
- Right ($x=1$): $\phi = 1$ (Dirichlet)
- Other faces: $\partial \phi / \partial n = 0$ (Neumann, zero flux)

The analytical solution is $\phi(x,y,z) = x$ (linear variation in x-direction).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyfvtool as pf

print('PyFVTool version:', pf.__version__)

## Step 1: Create a Simple Tetrahedral Mesh

We create a simple tetrahedralization of a unit cube using 8 nodes and 6 tetrahedra (standard subdivision of a cube into tetrahedra).

For production problems, you would use mesh generators like Gmsh, TetGen, or `scipy.spatial.Delaunay` for 3D.

In [ ]:
# Define nodes of a unit cube
nodes = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 1.0]
])

# Add interior point to avoid degenerate tetrahedra
nodes = np.vstack([nodes, [0.5, 0.5, 0.5]])

# Generate tetrahedral mesh using Delaunay triangulation
from scipy.spatial import Delaunay
tri = Delaunay(nodes)
cells = tri.simplices

print(f'Generated {len(cells)} tetrahedra')

# Create the mesh
print('Creating unstructured 3D mesh...')
mesh = pf.UnstructuredMesh3D(nodes, cells)
print(f'Mesh created: {mesh.num_cells} tetrahedra, {mesh.num_faces} triangular faces')

## Step 2: Tag Boundaries by Geometric Location

We identify boundary faces by their geometric location using a simple tolerance-based approach. Each face center is checked against the domain boundaries.

For more complex geometries, you would use the mesh generator's physical groups or a more sophisticated tagging function.

In [ ]:
# Tag boundaries by geometric location
boundary_tags = {}
tol = 1e-6
fc = mesh._face_centers

# Identify faces on each boundary
boundary_tags["left"] = np.where(fc[:, 0] < tol)[0]      # x=0
boundary_tags["right"] = np.where(fc[:, 0] > 1.0 - tol)[0]  # x=1
boundary_tags["bottom"] = np.where(fc[:, 1] < tol)[0]   # y=0
boundary_tags["top"] = np.where(fc[:, 1] > 1.0 - tol)[0]    # y=1
boundary_tags["front"] = np.where(fc[:, 2] < tol)[0]    # z=0
boundary_tags["back"] = np.where(fc[:, 2] > 1.0 - tol)[0]   # z=1

# Re-create mesh with these tags
mesh2 = pf.UnstructuredMesh3D(nodes, cells, boundary_tags)

print(f'Boundary tags: {list(boundary_tags.keys())}')
for tag, faces in boundary_tags.items():
    print(f'  {tag}: {len(faces)} faces')

## Step 3: Set Boundary Conditions

We set Dirichlet conditions on left and right faces, and Neumann (zero flux) on other faces.

In [ ]:
# Set up boundary conditions
print('Setting boundary conditions...')
BC = pf.BoundaryConditions(mesh2)

# Left: Dirichlet phi = 0
BC.left.a[:] = 0.0
BC.left.b[:] = 1.0
BC.left.c[:] = 0.0

# Right: Dirichlet phi = 1
BC.right.a[:] = 0.0
BC.right.b[:] = 1.0
BC.right.c[:] = 1.0

# Other faces: zero flux (default Neumann a=1, b=0, c=0)
print('  Left: Dirichlet phi=0')
print('  Right: Dirichlet phi=1')
print('  Other faces: Neumann zero flux (default)')

## Step 4: Solve the Diffusion Equation

We create cell variables, build the diffusion term, add boundary conditions, and solve the linear system.

In [ ]:
# Create cell variable with initial value
phi = pf.CellVariable(mesh2, 0.0, BC)

# Set diffusion coefficient
D = pf.CellVariable(mesh2, 1.0)
D_face = pf.harmonicMean(D)

# Build and solve the linear system
print('Building diffusion term...')
Mdiff = pf.diffusionTerm(D_face)
Mbc, RHSbc = pf.boundaryConditionsTerm(BC)

M = Mdiff + Mbc
RHS = RHSbc

print('Solving linear system...')
phi_sol = pf.solveMatrixPDE(mesh2, M, RHS)

print('Solution computed successfully!')

## Step 5: Visualize the Solution on Boundary Faces

3D visualization is more challenging. We show the solution on boundary faces using a 3D plot. Each boundary triangle is colored by the value of its owner cell.

In [ ]:
# Visualize solution on boundary faces
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Get boundary face information
boundary_faces = mesh2.boundary_faces
face_nodes = mesh2._face_nodes  # shape: (num_faces, 3) for triangular faces

# Plot each boundary face colored by owner cell value
for f in boundary_faces:
    # Get the three nodes of this triangular face
    node_indices = face_nodes[f]
    triangle = nodes[node_indices]  # shape: (3, 3)
    
    # Get the value from the owner cell
    owner_cell = mesh2.owner[f]
    cell_value = phi_sol.value[owner_cell]
    
    # Create polygon
    from matplotlib.collections import Poly3DCollection
    verts = [triangle]
    coll = Poly3DCollection(verts, alpha=0.8)
    coll.set_facecolor(plt.cm.viridis(cell_value))  # color by phi value
    ax.add_collection3d(coll)

# Set plot limits and labels
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_zlim([0, 1])
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Solution on Boundary Faces')

# Add colorbar
mappable = plt.cm.ScalarMappable(cmap='viridis')
mappable.set_array(phi_sol.value)
plt.colorbar(mappable, ax=ax, label=r'$\phi$', shrink=0.6)

plt.tight_layout()
plt.show()

## Step 6: Analyze Results

We compute statistics and compare with the analytical solution $\phi(x) = x$.

In [ ]:
# Compute and print some statistics
print('\nSolution statistics:')
print(f'  Min: {phi_sol.value.min():.4f}')
print(f'  Max: {phi_sol.value.max():.4f}')
print(f'  Mean: {phi_sol.value.mean():.4f}')

# Expected linear variation in x-direction
x_cells = mesh2._cell_centers[:, 0]
phi_expected = x_cells  # phi(x) = x for pure diffusion with these BCs
error = np.abs(phi_sol.value - phi_expected).max()
print(f'  Max error from linear solution phi(x)=x: {error:.4f}')
print(f'  Relative error: {error:.2%}')

## Summary

This example shows that PyFVTool's unstructured mesh support works for 3D tetrahedral meshes:

1. **Mesh creation**: Construct from nodes/cells arrays or use mesh generators
2. **Boundary tagging**: Use geometric or custom tagging
3. **Unified discretization**: Same `diffusionTerm()`, `boundaryConditionsTerm()` functions
4. **Visualization**: 3D boundary face rendering (slice planes would be needed for interior visualization)

## Next Steps

For more complex 3D problems:
- Use Gmsh, TetGen, or other mesh generators via the `from_gmsh()`, `from_meshpy()` methods
- Implement slice planes or volume rendering for interior visualization
- Try convection-diffusion problems with `convectionUpwindTerm()`
- Explore transient problems with `transientTerm()`